<strong><span style="color:Orange;font-size:40px">News Summarisation language model</strong><br>
<strong><span style="color:darkorange;font-size:20px"> Objectives</strong><br>
- develop a language model able to injest raw text format reviews, classify and summarise the review<br>
- Processing of raw input to classification of subject, sentiment ranking<br>
- recomend the next steps to rectify if a complaint

<span style="color:skyblue;font-size:20px"> Imports

In [35]:
import pandas as pd
import numpy as np
from transformers import pipeline, BartTokenizer, BartForConditionalGeneration
import spacy
import torch
import evaluate
import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.8"

device = "mps" if torch.backends.mps.is_available() else "cpu"


developmental_run = True

<span style="color:skyblue;font-size:20px"> Raw Data collection, Train, validate and Test

In [91]:
from datasets import load_dataset
dataset = load_dataset("cnn_dailymail", "3.0.0", trust_remote_code=True)

print(len(dataset["train"]))

# display the raw data structure
# print(dataset)
# display(dataset["train"][0])

287113


<span style="font-size:20px;color:orange"> Full/ Dev run

In [102]:
train_data = dataset["train"].to_pandas()

if developmental_run:
    train_data = train_data.head(20000)
    print(f"Dev Run: {train_data.shape}")
else:
    print(f"Full Run: {train_data.shape}")

print(train_data.columns)

Dev Run: (20000, 3)
Index(['article', 'highlights', 'id'], dtype='object')


<span style="color:skyblue;font-size:20px">Tokenisation ready for the transformer

<span style="font-size:20px;color:skyblue">Initiation of transformer - untrained

In [103]:
# summeriser = pipeline(
#     "summarization",
#     model="google/long-t5-tglobal-base",
#     device=device,
# )

# tokeniser = BartTokenizer.from_pretrained("facebook/bart-large")

# text = train_data['article'].tolist()
# # text = tokeniser
# summaries = []
# summeries = summeriser(
#     text,
#     # min_length=30,
#     do_sample=False,
# )
# for summary in summeries:
#     summaries.append(summary['summary_text'])
# train_data['summary'] = summaries
# train_data = train_data.rename(columns={'highlights': 'target'})

In [104]:
# display(train_data.head())

In [105]:
# rouge = evaluate.load("rouge")
# results = rouge.compute(
#     predictions=train_data['summary'].tolist(),
#     references=train_data['target'].tolist(),
#     use_stemmer=True,
# )
# print(f"ROUGE-1: {results['rouge1']}")

<span style="font-size:20px;color:skyblue">Training regieme for fine tuning

In [107]:
import torch
from transformers import LongT5ForConditionalGeneration, AutoTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, AutoModelForSeq2SeqLM
from datasets import Dataset
import gc
from itertools import product
from rouge_score import rouge_scorer
from tqdm import tqdm
# Select Device (MPS for Mac, CPU as fallback)
device = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Using device: {device}")



# Preprocessing Function (Reducing Token Count to Manage Memory)
def preprocessbatch(batch):
    inputs = tokenizer(batch['article'], max_length=256, truncation=True, padding="max_length")
    target = tokenizer(batch['highlights'], max_length=128, truncation=True, padding="max_length")
    inputs['labels'] = target['input_ids']
    
    # Preserve original text for evaluation
    inputs['article'] = batch['article']
    inputs['highlights'] = batch['highlights']

    return inputs

def compute_rouge(preds, refs):
    if not preds or not refs or len(preds) != len(refs):
        return {metric: 0.0 for metric in ['rouge1', 'rouge2', 'rougeL']}

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = [scorer.score(pred, ref) for pred, ref in zip(preds, refs)]

    avg_scores = {
        metric: sum(score[metric].fmeasure for score in scores) / max(len(scores), 1)
        for metric in ['rouge1', 'rouge2', 'rougeL']
    }

    return avg_scores

# Tokenizing Data
tokenised_df = train_data.apply(lambda row: preprocessbatch(row), axis=1).apply(pd.Series)


# Convert DataFrame to Hugging Face Dataset
tokenised_ds = Dataset.from_pandas(tokenised_df)


params_to_test = {
    "learning_rate": [3e-5],
    "per_device_train_batch_size": [5],
    "num_train_epochs": [3, 5],
    "weight_decay": [ 0.1],
}
# Get all possible combinations
grid_combinations = list(product(*params_to_test.values()))

best_score = None
best_params = None

for values in grid_combinations:
    model_name = 't5-small'

    # Load Model and Tokenizer
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = model.to(device)
    
    params = dict(zip(params_to_test.keys(), values))
    print(f"\n🔧 Testing hyperparams: {params}")

    args = Seq2SeqTrainingArguments(
        output_dir="long-t5-summarisation",
        learning_rate=params["learning_rate"],
        per_device_train_batch_size=params["per_device_train_batch_size"],
        num_train_epochs=params["num_train_epochs"],
        weight_decay=params["weight_decay"],
        eval_strategy="epoch",
        save_strategy="epoch",
        report_to="none",
        load_best_model_at_end=True,
        logging_steps=50,
    )

    trainer = Seq2SeqTrainer(
        model=model.to(device),  # Ensure CPU usage
        args=args,
        train_dataset=tokenised_ds,
        eval_dataset=tokenised_ds,
        tokenizer=tokenizer,
    )

    print("🚂 Starting training...")
    trainer.train()
    print("✅ Training complete.")

    # Evaluation
    print("🔍 Evaluating on sample data...\n")
    sample_articles = tokenised_ds["article"][:5]
    ground_truths = tokenised_ds["highlights"][:5]
    predictions = []
    rouge_scores = []


    for i, (text, reference) in enumerate(tqdm(zip(sample_articles, ground_truths), total=5)):
        # Tokenise and move all input data to the same device
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Generate summary from the model
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=256,
                num_beams=4,
            )

        # Decode and save prediction
        decoded = tokenizer.decode(outputs[0].detach().cpu(), skip_special_tokens=True)
        predictions.append(decoded)

        rouge_result = compute_rouge([decoded], [reference])
        rouge_l = rouge_result["rougeL"]
        rouge_scores.append(rouge_l)

        print(f"\n📄 Article {i+1}")
        print(f"🧠 Prediction: {decoded}")
        print(f"✅ Reference: {reference}")
        print(f"📊 ROUGE-L: {rouge_l:.4f}")

        del inputs, outputs
        if device == "mps":
            torch.mps.empty_cache()
            gc.collect()

    avg_rouge = np.mean(rouge_scores)
    print(f"\n📈 Average ROUGE-L: {avg_rouge:.4f}")

    if best_score is None or avg_rouge > best_score:
        best_score = avg_rouge
        best_params = params

print(f"\n🏁 Best Params: {best_params} with ROUGE-L Score: {best_score:.4f}")

Using device: mps

🔧 Testing hyperparams: {'learning_rate': 3e-05, 'per_device_train_batch_size': 5, 'num_train_epochs': 3, 'weight_decay': 0.1}
🚂 Starting training...


/var/folders/f2/_cjvdffn2x91f953fzbj08dc0000gn/T/ipykernel_20406/746242180.py:85: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,1.226500,1.090486
2,1.126100,1.077389
3,1.179300,1.072702


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


✅ Training complete.
🔍 Evaluating on sample data...



...


📄 Article 3
🧠 Prediction: Dr. John Hink, an emergency room physician, jumped into his car and rushed to the scene. He rushed to the scene in 15 minutes; 55 people got into ambulances in less than two hours. "I could see the whole bridge as it was going down, as it was going down," he says.
✅ Reference: NEW: "I thought I was going to die," driver says .
Man says pickup truck was folded in half; he just has cut on face .
Driver: "I probably had a 30-, 35-foot free fall"
Minnesota bridge collapsed during rush hour Wednesday .
📊 ROUGE-L: 0.1075


 80%|████████  | 4/5 [00:10<00:02,  2.32s/it]


📄 Article 4
🧠 Prediction: NEW: "None appeared worrisome," White House spokesman says. NEW: All were small, less than a centimeter [half an inch] in diameter. NEW: Bush is in good humor and will resume his activities at Camp David. NEW: President's last colonoscopy was in June 2002.
✅ Reference: Five small polyps found during procedure; "none worrisome," spokesman says .
President reclaims powers transferred to vice president .
Bush undergoes routine colonoscopy at Camp David .
📊 ROUGE-L: 0.2353

📄 Article 5
🧠 Prediction: NFL commissioner: "Your admitted conduct was not only illegal, but also cruel and reprehensible" Vick admitted to participating in a dogfighting ring as part of a plea deal with federal prosecutors. Falcons could "assert any claims or remedies" to recover $22 million of Vick's signing bonus.
✅ Reference: NEW: NFL chief, Atlanta Falcons owner critical of Michael Vick's conduct .
NFL suspends Falcons quarterback indefinitely without pay .
Vick admits funding dogfighting

100%|██████████| 5/5 [00:12<00:00,  2.53s/it]


📈 Average ROUGE-L: 0.2524

🏁 Best Params: {'learning_rate': 3e-05, 'per_device_train_batch_size': 5, 'num_train_epochs': 5, 'weight_decay': 0.1} with ROUGE-L Score: 0.2524


In [108]:
print(f"\n🏁 Best Params: {best_params} with ROUGE-L Score: {best_score:.4f}")


🏁 Best Params: {'learning_rate': 3e-05, 'per_device_train_batch_size': 5, 'num_train_epochs': 5, 'weight_decay': 0.1} with ROUGE-L Score: 0.2524
